# FAKE NEWS DETECTION AI — Academic ML Pipeline Notebook
**Project Title:** End-to-End Fake News Text Classification using TF-IDF and Supervised Machine Learning
**Author:** Academic Internship Student Project

---
## 1. Project Overview
This notebook demonstrates the complete step-by-step machine learning workflow:
1. Data Acquisition & Inspection
2. NLP Text Preprocessing
3. Stratified Train/Test Split (80/20)
4. Feature Engineering using TF-IDF Vectorization
5. Training Supervised Classifiers (Logistic Regression, Naive Bayes, Linear SVM)
6. Metric Evaluation (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
7. Top Predictive Feature Extraction
8. Model Persistence (`joblib` serialization)

In [1]:
# Step 1: Import Core Libraries
import os
import json
import re
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

print('All required libraries imported successfully!')

All required libraries imported successfully!


## 2. Load & Inspect Dataset

In [2]:
dataset_path = os.path.join('..', 'backend', 'data', 'news_dataset.csv')
df = pd.read_csv(dataset_path)
print(f'Dataset Shape: {df.shape}')
print('\nClass Distribution:')
print(df['label'].value_counts())
df.head()

Dataset Shape: (4056, 5)

Class Distribution:
label
REAL    2124
FAKE    1932
Name: count, dtype: int64


## 3. NLP Text Cleaning Pipeline

In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['combined_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['cleaned_text'] = df['combined_text'].apply(clean_text)
df = df[df['cleaned_text'].str.strip() != ''].drop_duplicates(subset=['cleaned_text']).reset_index(drop=True)
print(f'Cleaned dataset count: {len(df)} samples')

Cleaned dataset count: 3909 samples


## 4. Train/Test Split & TF-IDF Vectorization

In [4]:
label_map = {'FAKE': 0, 'REAL': 1}
y = df['label'].map(label_map).values
X = df['cleaned_text'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    min_df=2,
    max_df=0.95
)

# Fit strictly on training set
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f'Training Samples: {X_train_tfidf.shape[0]}')
print(f'Testing Samples: {X_test_tfidf.shape[0]}')
print(f'TF-IDF Feature Vocabulary: {len(vectorizer.vocabulary_)} tokens')

Training Samples: 3127
Testing Samples: 782
TF-IDF Feature Vocabulary: 5000 tokens


## 5. Model Training & Evaluation

In [5]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(alpha=1.0),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42), cv=5)
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary')
    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}
    print(f'=== {name} ===')
    print(classification_report(y_test, preds, target_names=['FAKE', 'REAL']))

=== Logistic Regression ===
              precision    recall  f1-score   support

        FAKE       0.78      0.67      0.72       375
        REAL       0.73      0.83      0.78       407

    accuracy                           0.75       782
   macro avg       0.76      0.75      0.75       782
weighted avg       0.76      0.75      0.75       782

=== Multinomial Naive Bayes ===
              precision    recall  f1-score   support

        FAKE       0.76      0.75      0.75       375
        REAL       0.77      0.78      0.78       407

    accuracy                           0.77       782
   macro avg       0.77      0.77      0.77       782
weighted avg       0.77      0.77      0.77       782

=== Linear SVM ===
              precision    recall  f1-score   support

        FAKE       0.75      0.72      0.73       375
        REAL       0.75      0.78      0.77       407

    accuracy                           0.75       782
   macro avg       0.75      0.75      0.75      

## 6. Model Comparison Table

In [6]:
results_df = pd.DataFrame(results).T
print(results_df)

                         Accuracy  Precision    Recall  F1-Score
Logistic Regression      0.753197   0.733624  0.825553  0.776879
Multinomial Naive Bayes  0.765985   0.770531  0.783784  0.777101
Linear SVM               0.751918   0.750588  0.783784  0.766827


## 7. Model Serialization

In [7]:
# Select best model dynamically based on highest F1-Score
best_name = max(results, key=lambda k: results[k]['F1-Score'])
best_model = models[best_name]
print(f'Selected Best Model: {best_name}')

output_dir = os.path.join('..', 'backend', 'models')
os.makedirs(output_dir, exist_ok=True)

joblib.dump(best_model, os.path.join(output_dir, 'best_model.joblib'))
joblib.dump(vectorizer, os.path.join(output_dir, 'tfidf_vectorizer.joblib'))
print('Trained model artifacts successfully serialized!')

Selected Best Model: Multinomial Naive Bayes
Trained model artifacts successfully serialized!
